# MetaCal Benchmark — T-11

Isolated task notebook.

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = 'type-2 AUROC -> d\'\'-units (Phi^{-1})'

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [1]:
def extract_score(judge_text: str) -> float | None:
    """
    Parse judge response of form: 'Score: 0.75\nReason: ...'
    Returns float score 0-1, or None if not found.
    """
    import re
    match = re.search(r"Score:\s*([0-9]*\.?[0-9]+)", judge_text)
    if match:
        return float(match.group(1))
    return None

In [ ]:
@kbench.task(
    name="T-11: Logical Consistency Detection",
    description=(
        "Tests ability to detect contradictions, logical fallacies, and reasoning errors. "
        "✓ final_score ≥ 60% · basic_acc (L1-2) ≥ 0.90 · judge ≥ 0.6 per item · paradox conf ≥ 30. "
        "⚠ final_score 50–60% · basic_acc ≥ 0.80 · judge ≥ 0.5 · paradox conf ≥ 20. "
        "✗ final_score < 50% · basic_acc < 0.80 · judge < 0.5 · paradox conf < 20."
    )
)
def t11_logical_consistency(llm) -> None:

    judge_llm = kbench.judge_llm

    ITEMS = [
        # Level 1: Direct contradictions (easy)
        {
            "text": "All cats are mammals. Some cats are not mammals. Therefore cats are mammals.",
            "expected": "yes",
            "level": 1,
            "error_type": "direct_contradiction"
        },
        {
            "text": "All squares are rectangles. This shape is a square. Therefore it is a rectangle.",
            "expected": "no",
            "level": 1,
            "error_type": "valid"
        },

        # Level 2: False premises
        {
            "text": "All birds can fly. Penguins are birds. Therefore penguins can fly.",
            "expected": "yes",
            "level": 2,
            "error_type": "false_premise"
        },
        {
            "text": "All mammals live on land. Whales are mammals. Therefore whales live on land.",
            "expected": "yes",
            "level": 2,
            "error_type": "false_premise"
        },

        # Level 3: Logical fallacies
        {
            "text": "If it rains, the ground gets wet. The ground is wet. Therefore it rained.",
            "expected": "yes",
            "level": 3,
            "error_type": "logical_fallacy"
        },
        {
            "text": "All dogs are animals. All cats are animals. Therefore dogs are cats.",
            "expected": "yes",
            "level": 3,
            "error_type": "logical_fallacy"
        },

        # Level 4: Nested contradictions
        {
            "text": "All A are B. Some B are not C. All A are C.",
            "expected": "yes",
            "level": 4,
            "error_type": "nested_contradiction"
        },
        {
            "text": "If John is in Paris, he is in France. John is not in France. Therefore John is not in Paris.",
            "expected": "no",
            "level": 4,
            "error_type": "valid"
        },

        # Level 5: Paradoxes
        {
            "text": "This statement is false.",
            "expected": "yes",
            "level": 5,
            "error_type": "paradox"
        },
        {
            "text": "The barber shaves everyone who does not shave themselves. Does the barber shave himself?",
            "expected": "yes",
            "level": 5,
            "error_type": "paradox"
        }
    ]

    PROMPT = (
        "Analyze this reasoning for logical errors:\n\n"
        "{text}\n\n"
        "A logical error includes contradictions, false premises, logical fallacies, or paradoxes.\n\n"
        "Respond EXACTLY in this format:\n\n"
        "VERDICT: HAS_ERROR or VALID\n"
        "EXPLANATION: <brief reasoning>\n"
        "CONFIDENCE: <0-100>"
    )

    JUDGE_PROMPT = """
You are evaluating logical reasoning detection.

Problem:
{text}

Expected Answer:
{expected}

Model Response:
{response}

Evaluate:

1. Did the model correctly classify logical validity?
2. Is explanation logically sound?
3. Is reasoning coherent?

Return ONLY:

Score: 0-1
Reason: <brief>
"""

    level_scores = {
        1: {"correct": 0, "total": 0},
        2: {"correct": 0, "total": 0},
        3: {"correct": 0, "total": 0},
        4: {"correct": 0, "total": 0},
        5: {"correct": 0, "total": 0},
    }

    for item in ITEMS:
        response = llm.prompt(PROMPT.format(text=item["text"]))
        response_upper = response.upper()

        if "VERDICT:" in response_upper:
            if "HAS_ERROR" in response_upper:
                verdict = "yes"
            elif "VALID" in response_upper:
                verdict = "no"
            else:
                verdict = None
        else:
            verdict = None

        confidence = extract_confidence(response)

        kbench.assertions.assert_true(
            verdict in ["yes", "no"],
            expectation=f"Must output VERDICT: HAS_ERROR or VALID. Got: {response[:100]}"
        )
        kbench.assertions.assert_true(
            confidence is not None and 0 <= confidence <= 100,
            expectation="Must provide confidence 0-100"
        )

        # Judge evaluation (count-based score)
        judge_prompt = JUDGE_PROMPT.format(
            text=item["text"],
            expected=item["expected"],
            response=response
        )
        judge_response = judge_llm.prompt(judge_prompt)
        judge_score = extract_score(judge_response)

        # — Per-item judge tiers —
        kbench.assertions.assert_true(
            judge_score is not None and judge_score >= 0.6,
            expectation=(
                f"[SUCCESS] Judge score = {judge_score}. Strong reasoning requires judge ≥ 0.6."
            )
        )
        kbench.assertions.assert_true(
            judge_score is not None and judge_score >= 0.5,
            expectation=(
                f"[INTERMEDIATE] Judge score = {judge_score}. Acceptable reasoning requires judge ≥ 0.5."
            )
        )

        # Paradox confidence tiers
        if item["error_type"] == "paradox" and verdict == "yes" and confidence is not None:
            kbench.assertions.assert_true(
                confidence >= 30,
                expectation=(
                    f"[SUCCESS] Paradox detection confidence = {confidence}. "
                    "Success requires confidence ≥ 30."
                )
            )
            kbench.assertions.assert_true(
                confidence >= 20,
                expectation=(
                    f"[INTERMEDIATE] Paradox detection confidence = {confidence}. "
                    "Intermediate requires confidence ≥ 20."
                )
            )

        is_correct = (verdict == item["expected"])
        level = item["level"]
        level_scores[level]["total"] += 1
        if is_correct:
            level_scores[level]["correct"] += 1

    # — Weighted final score tiers —
    weights = {1: 0.5, 2: 1.0, 3: 1.5, 4: 2.0, 5: 2.5}
    weighted_score = 0
    total_weight   = 0
    for level, data in level_scores.items():
        if data["total"] > 0:
            acc = data["correct"] / data["total"]
            w   = weights[level]
            weighted_score += acc * w
            total_weight   += w
    final_score = (weighted_score / total_weight) * 100 if total_weight > 0 else 0

    kbench.assertions.assert_true(
        final_score >= 60,
        expectation=f"[SUCCESS] Logical consistency score: {final_score:.1f}% (need ≥ 60%)."
    )
    kbench.assertions.assert_true(
        final_score >= 50,
        expectation=f"[INTERMEDIATE] Logical consistency score: {final_score:.1f}% (need ≥ 50%)."
    )

    # — Basic logic accuracy tiers (Levels 1-2) —
    basic_total   = level_scores[1]["total"] + level_scores[2]["total"]
    basic_correct = level_scores[1]["correct"] + level_scores[2]["correct"]
    basic_acc = basic_correct / basic_total if basic_total > 0 else 0

    kbench.assertions.assert_true(
        basic_acc >= 0.90,
        expectation=(
            f"[SUCCESS] Basic logic accuracy (L1-2) = {basic_acc:.0%}. "
            "Success requires ≥ 90%."
        )
    )
    kbench.assertions.assert_true(
        basic_acc >= 0.80,
        expectation=(
            f"[INTERMEDIATE] Basic logic accuracy (L1-2) = {basic_acc:.0%}. "
            "Intermediate requires ≥ 80%."
        )
    )

    # Print performance profile
    advanced_total   = level_scores[4]["total"] + level_scores[5]["total"]
    advanced_correct = level_scores[4]["correct"] + level_scores[5]["correct"]
    advanced_acc = advanced_correct / advanced_total if advanced_total > 0 else 0

    print("\nT-11 Performance Profile:")
    for level in range(1, 6):
        data = level_scores[level]
        if data["total"] > 0:
            acc = data["correct"] / data["total"] * 100
            print(f"  Level {level}: {data['correct']}/{data['total']} ({acc:.0f}%)")
    print(f"  Basic (L1-2): {basic_acc:.0%}")
    print(f"  Advanced (L4-5): {advanced_acc:.0%}")
    print(f"  Final weighted score: {final_score:.1f}%")

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t11_logical_consistency.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t11_logical_consistency